# Procesando texto: resúmenes de noticias con un LLM

Para cada una de las 10 noticias de la tabla `noticias` generamos un resumen de **máximo 50 palabras**. Usamos el modelo `google/gemini-2.5-flash-lite` a través del endpoint `responses` de OpenRouter, **con el razonamiento desactivado** para minimizar costos.

**La API key** no se escribe en el notebook. Se lee de la variable de entorno `OPENROUTER_API_KEY`, o de un archivo `.env` en esta carpeta, que está en `.gitignore`. Así la clave no termina en el repositorio público.

**Decisiones de diseño:**

- **Respetar el límite de 50 palabras.** El LLM no cuenta palabras con exactitud, así que pedimos un objetivo con holgura (~40 palabras) y **verificamos** el largo de cada resumen. Si alguno se pasa, lo pedimos de nuevo con una instrucción más estricta. Solo como último recurso lo truncamos, y lo dejamos marcado para que se note.
- **Resultados reproducibles.** Usamos `temperature=0`, un `max_output_tokens` acotado y reintentos ante errores de red.

In [1]:
import os
import time
from pathlib import Path

import mysql.connector
import pandas as pd
import requests

MODELO = "google/gemini-2.5-flash-lite"
URL = "https://openrouter.ai/api/v1/responses"
MAX_PALABRAS = 50

In [2]:
def validation_gate(condicion: bool, mensaje: str) -> None:
    """Puerta de validación: detiene el notebook si un supuesto no se cumple."""
    if not condicion:
        raise AssertionError(f"GATE FALLIDO: {mensaje}")
    print(f"OK  {mensaje}")

In [3]:
def get_api_key() -> str | None:
    if os.environ.get("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"]
    env = Path(".env")
    if env.exists():
        for linea in env.read_text().splitlines():
            if linea.startswith("OPENROUTER_API_KEY="):
                return linea.split("=", 1)[1].strip().strip('"').strip("'")
    return None


API_KEY = get_api_key()
validation_gate(bool(API_KEY), f"hay una API key de OpenRouter disponible: {API_KEY[:4]}...{API_KEY[-4:]}")

OK  hay una API key de OpenRouter disponible: sk-o...8c17


## 1. Descarga de la tabla `noticias`

In [4]:
DB = dict(
    host="sql.freedb.tech",
    port=3306,
    user="u_y8tlWZ",
    password="wSkxw3xbhsQg",
    database="freedb_N4nnid0a",
)


def get_connection(n_retries: int = 6):
    """La BD es compartida y limita las conexiones simultáneas: reintentamos con espera creciente."""
    for intento in range(1, n_retries + 1):
        try:
            return mysql.connector.connect(**DB)
        except mysql.connector.Error as e:
            if intento == n_retries:
                raise
            print(
                f"Conexión rechazada ({e.msg}); reintento {intento} en {5 * intento}s"
            )
            time.sleep(5 * intento)

In [5]:
df_news = pd.read_sql("SELECT * FROM noticias", get_connection())

/tmp/ipykernel_193318/2170744738.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_news = pd.read_sql("SELECT * FROM noticias", get_connection())


In [6]:
df_news.to_parquet("noticias.parquet", index=False)

In [7]:
df_news.assign(palabras_cuerpo=df_news["cuerpo"].str.split().str.len())[
    ["titular", "palabras_cuerpo"]
]

,titular,palabras_cuerpo
0,escenas de geometría variable,613
1,iglesias se reunirá con junqueras sin cristal ...,249
2,bankia paga otros 219 millones al estado vía d...,312
3,sánchez tiene ya una mayoría en el congreso pa...,621
4,un presidente irresponsable,570
5,castro embarga el palacete y otra finca a mata...,632
6,"el jefe del ejecutivo, dispuesto a reunirse co...",224
7,el último choque con puigdemont y las diferenc...,921
8,interior jubila al policía que perdió la visió...,846
9,"otegi sobre bateragune ""ni nos domaron, ni nos...",349


## 2. Llamada al modelo

La consigna indica que el texto viene en `response.json()["output"][0]["content"][0]["text"]`. Según la documentación de OpenRouter, el arreglo `output` también puede traer elementos de tipo `reasoning`. Por eso extraemos el texto filtrando los elementos de tipo `message`, que funciona en ambos casos.

Además verificamos que la respuesta indique **0 tokens de razonamiento**, para confirmar que el razonamiento realmente quedó desactivado.

In [8]:
PROMPT = (
    "Resume la siguiente noticia en español, en un solo párrafo de máximo {limite} palabras. "
    "Incluye solo los hechos principales (quién, qué, cuándo). "
    "Responde únicamente con el resumen, sin título, viñetas ni comillas.\n\n"
    "Noticia:\n{texto}"
)

In [9]:
def extract_text(datos: dict) -> str:
    return "".join(
        parte.get("text", "")
        for item in datos.get("output", [])
        if item.get("type") == "message"
        for parte in item.get("content", [])
        if parte.get("type") == "output_text"
    ).strip()

In [10]:
REINTENTABLES = {429, 500, 502, 503, 504}  # límite de tasa o errores transitorios del proveedor


def call_llm(prompt: str, n_retries: int = 6) -> dict:
    """Llama al endpoint responses. Reintenta con espera exponencial ante 429/5xx."""
    for intento in range(1, n_retries + 1):
        try:
            r = requests.post(
                URL,
                headers={
                    "Authorization": f"Bearer {API_KEY}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": MODELO,
                    "input": prompt,
                    "reasoning": {"effort": "none"},  # desactiva el razonamiento
                    "temperature": 0,
                    "max_output_tokens": 200,
                },
                timeout=60,
            )
        except requests.ConnectionError as e:  # red caída: también es transitorio
            espera, motivo = 2**intento, str(e)
        else:
            if r.ok:
                return r.json()
            if r.status_code not in REINTENTABLES:  # 400/401/402: reintentar no sirve
                raise RuntimeError(f"OpenRouter respondió {r.status_code}: {r.text[:500]}")
            espera = float(r.headers.get("Retry-After", 2**intento))
            motivo = f"HTTP {r.status_code}"
        if intento == n_retries:
            raise RuntimeError(f"OpenRouter no respondió tras {n_retries} intentos ({motivo})")
        print(f"Intento {intento} falló ({motivo}); reintento en {espera:.0f}s")
        time.sleep(espera)

In [11]:
def resumir(texto: str) -> dict:
    """Devuelve el resumen y metadatos de control."""
    limite_pedido = 50
    reintentos_largo = 0

    while True:
        datos = call_llm(prompt=PROMPT.format(limite=limite_pedido, texto=texto))
        resumen = extract_text(datos)
        razonamiento = (datos.get("usage", {}).get("output_tokens_details") or {}).get(
            "reasoning_tokens", 0
        ) or 0

        if len(resumen.split()) <= MAX_PALABRAS or reintentos_largo >= 1:
            break

        limite_pedido, reintentos_largo = (
            30,
            reintentos_largo + 1,
        )  # instrucción más estricta
    truncado = len(resumen.split()) > MAX_PALABRAS
    if truncado:  # último recurso
        resumen = " ".join(resumen.split()[:MAX_PALABRAS])
    return {
        "resumen": resumen,
        "tokens_razonamiento": razonamiento,
        "truncado": truncado,
        "status": datos.get("status"),
    }

**Prueba con una sola noticia** antes de procesar todo el lote, para validar la integración y la forma de la respuesta:

In [26]:
prueba = call_llm(PROMPT.format(limite=40, texto=df_news.loc[0, "cuerpo"]))
validation_gate(
    prueba.get("status") == "completed", "la respuesta de prueba está completa"
)
validation_gate(bool(extract_text(prueba)), "la respuesta contiene texto")
print(extract_text(prueba))

OK  la respuesta de prueba está completa
OK  la respuesta contiene texto
Zapatero usó "geometría variable" para pactos cambiantes. Sánchez aplica esta estrategia con diversos socios, buscando estabilidad a pesar de la complejidad y posibles crisis futuras.


## 3. Resúmenes de todas las noticias

In [27]:
resultados = pd.DataFrame([resumir(t) for t in df_news["cuerpo"]])
df_news = pd.concat([df_news, resultados], axis=1)
df_news["palabras_resumen"] = df_news["resumen"].str.split().str.len()

validation_gate(
    len(df_news) == 10 and df_news["resumen"].str.len().gt(0).all(),
    "hay un resumen no vacío por noticia",
)
validation_gate(
    df_news["palabras_resumen"].le(MAX_PALABRAS).all(),
    f"ningún resumen supera {MAX_PALABRAS} palabras",
)
validation_gate(
    df_news["tokens_razonamiento"].eq(0).all(),
    "el razonamiento estuvo desactivado (0 tokens de razonamiento)",
)
print(f"Resúmenes truncados como último recurso: {df_news['truncado'].sum()}")

OK  hay un resumen no vacío por noticia
OK  ningún resumen supera 50 palabras
OK  el razonamiento estuvo desactivado (0 tokens de razonamiento)
Resúmenes truncados como último recurso: 0


In [29]:
pd.set_option("display.max_colwidth", None)
df_news[["titular", "resumen", "palabras_resumen", "cuerpo"]]

titular  \
0                                                                              escenas de geometría variable   
1                                       iglesias se reunirá con junqueras sin cristal y sin límite de tiempo   
2                            bankia paga otros 219 millones al estado vía dividendo, el mayor de su historia   
3            sánchez tiene ya una mayoría en el congreso para derogar la reforma laboral y la "ley mordaza "   
4                                                                                un presidente irresponsable   
5                              castro embarga el palacete y otra finca a matas para cubrir su fianza en nóos   
6                               el jefe del ejecutivo, dispuesto a reunirse con pablo iglesias si se lo pide   
7  el último choque con puigdemont y las diferencias entre defensas la cuenta atrás para el juicio al procés   
8                            interior jubila al policía que perdió la visión tras los disturbios del procés»   
9                          otegi sobre bateragune "ni nos domaron, ni nos doblaron, ni nos van a domesticar"   

                                                                                                                                                                                                                                                                                                                resumen  \
0                                                            El presidente Sánchez utiliza la "geometría variable" para gobernar, buscando apoyos puntuales de diversos partidos. Esta estrategia, iniciada por Zapatero, busca flexibilidad ante la falta de pactos estables, aunque históricamente resultó dramática.   
1                                                                                                                                   Pablo Iglesias visitará a Oriol Junqueras este viernes en la cárcel de Lledoners para discutir la situación de los presos soberanistas y buscar el apoyo de ERC a los presupuestos.   
2                                                                                                                                                                             Bankia abonará este jueves 11 de abril su quinto dividendo, 219 millones de euros al Estado, sumando 961 millones devueltos por esta vía.   
3  La nueva composición del Congreso, con mayoría de izquierdas, permitirá al PSOE tramitar leyes progresistas como la eutanasia y derogar la reforma laboral. La investidura de Sánchez se prevé en junio/julio, con propuestas llegando en septiembre, priorizando presupuestos y la eliminación del veto del Senado.   
4                                                                                                                   El rey emérito Juan Carlos I regularizó ante Hacienda casi 4,5 millones de euros por viajes en jet privado pagados por una fundación, tras haber abonado 680.000 euros en diciembre por donaciones.   
5                                                                                                                                                                                 Hoy, el juez José Castro ordenó el embargo de bienes de Jaume Matas y otros acusados para cubrir fianzas millonarias en el caso Nóos.   
6                                                                                                                                             Mariano Rajoy, presidente del gobierno y del PP, se mostró dispuesto a reunirse con Pablo Iglesias, líder de Podemos, si este lo solicita, señalando que habla con todos.   
7                                                                                             El juicio al procés comenzará la semana del 5 de febrero en el Tribunal Supremo, con 12 acusados. La sentencia se espera para verano. El proceso coincide con negociaciones presupuestarias y tensiones independentistas.   
8                                

## 4. Comentarios

- **Control de calidad.** Los *gates* verifican lo que se puede verificar automáticamente: que haya un resumen por noticia, que respete el largo y que no haya habido razonamiento. **La fidelidad del resumen no se puede verificar así.** El modelo podría omitir un hecho clave o, más grave, agregar uno que no está en la noticia. Con 10 noticias lo razonable es revisarlas a mano comparando cada resumen con el titular y el cuerpo.
- **A mayor escala**, se podría usar una revisión por muestreo, o un segundo modelo como evaluador que verifique que cada afirmación del resumen está respaldada por el texto.
- **Costo.** Cada noticia es una llamada con unos cientos de tokens de entrada y menos de 100 de salida, en un modelo económico y sin razonamiento. Para volúmenes grandes convendría paralelizar las llamadas, respetando los límites de la API, y guardar las respuestas para no repetir llamadas al re-ejecutar.